# Agentic AI

## 실습 전 Tavily API 셋팅
- https://tavily.com 에서 발급 (무료 크레딧 제공)

`.env`

```
TAVILY_API_KEY=tvly-...
````

## Step 01 · 환경 설정

In [ ]:
%pip install -U langchain-tavily ddgs requests langchain-community

In [2]:
import json
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

print("OpenAI:", "OK" if os.getenv("OPENAI_API_KEY") else "없음", "| Tavily:", "OK" if os.getenv("TAVILY_API_KEY") else "없음")

MODEL = "gpt-6-luna"
llm = ChatOpenAI(model=MODEL)

OpenAI: OK | Tavily: OK


## Step 02 · 사용자 도구 정의

### 2-1. @tool
- 타입 힌트 = 입력 스키마
- docstring = 도구 설명

In [35]:
from langchain.tools import tool

from datetime import date, timedelta

@tool
def add_business_days(start_date: str, days: int) -> str:
    """시작 날짜(YYYY-MM-DD)에서 주말을 제외한 영업일 기준으로 days일 뒤의 날짜를 계산합니다."""

    d = date.fromisoformat(start_date)
    added = 0
    while added < days:
        d += timedelta(days=1)
        if d.weekday() < 5:
            added += 1
    return d.isoformat()

In [36]:
print("name       :", add_business_days.name)
print("description:", add_business_days.description)
print("args       :", add_business_days.args)
print("invoke     :", add_business_days.invoke({"start_date": "2026-10-02", "days": 3}))

name       : add_business_days
description: 시작 날짜(YYYY-MM-DD)에서 주말을 제외한 영업일 기준으로 days일 뒤의 날짜를 계산합니다.
args       : {'start_date': {'title': 'Start Date', 'type': 'string'}, 'days': {'title': 'Days', 'type': 'integer'}}
invoke     : 2026-10-07


### 2-2. 이름 · 설명 직접 지정

In [37]:
@tool("unit_converter", description="길이 단위를 변환합니다. km, m, mile 사이의 변환에 사용하세요.")
def convert(value: float, from_unit: str, to_unit: str) -> str:
    """(description 인자가 docstring보다 우선)"""
    to_m = {"km": 1000, "m": 1, "mile": 1609.344}
    return f"{value * to_m[from_unit] / to_m[to_unit]:.3f} {to_unit}"

In [38]:
print(convert.name, "|", convert.description)

unit_converter | 길이 단위를 변환합니다. km, m, mile 사이의 변환에 사용하세요.


In [39]:
print(convert.invoke({"value": 10, "from_unit": "km", "to_unit": "mile"}))

6.214 mile


### 2-3. args_schema 

- Pydantic으로 입력 형식 · 허용값 정의

In [40]:
from typing import Literal
from pydantic import BaseModel, Field

class ExchangeInput(BaseModel):
    amount: float = Field(description="변환할 금액")
    currency: Literal["USD", "JPY", "EUR"] = Field(description="변환할 외화 통화 코드")

RATES = {"USD": 1400.0, "JPY": 9.3, "EUR": 1510.0}   # 예시 환율 (원)

In [41]:
@tool(args_schema=ExchangeInput)
def to_krw(amount: float, currency: str) -> dict:
    """외화 금액을 원화로 환산합니다. 결과는 dict로 반환합니다."""
    return {
        "amount": amount, 
        "currency": currency, 
        "krw": round(amount * RATES[currency]), 
        "rate": RATES[currency]
    }

In [42]:
print(json.dumps(to_krw.args, ensure_ascii=False, indent=1))

{
 "amount": {
  "description": "변환할 금액",
  "title": "Amount",
  "type": "number"
 },
 "currency": {
  "description": "변환할 외화 통화 코드",
  "enum": [
   "USD",
   "JPY",
   "EUR"
  ],
  "title": "Currency",
  "type": "string"
 }
}


In [43]:
print(to_krw.invoke({"amount": 50, "currency": "USD"}))

{'amount': 50.0, 'currency': 'USD', 'krw': 70000, 'rate': 1400.0}


## Step 03 · Tool Calling 루프 

(에이전트 없이 직접 적용하는 방법)

1. `bind_tools`로 모델에 도구 목록 전달 → 
2. 모델이 `tool_calls` 생성 → 
3. 코드가 도구 실행 → 
4. `ToolMessage`로 결과 전달 → 
5. 모델이 최종 답변


In [44]:
from langchain.messages import HumanMessage, ToolMessage

tools = [add_business_days, convert, to_krw]
tools_by_name = {t.name: t for t in tools} # { str: func } 
llm_with_tools = llm.bind_tools(tools)

In [45]:
messages = [
    HumanMessage("2026-10-02부터 영업일 5일 뒤는 며칠이야? 그리고 30달러는 원화로 얼마야?")
]
ai = llm_with_tools.invoke(messages)
print("tool_calls:", ai.tool_calls)

tool_calls: [{'name': 'add_business_days', 'args': {'start_date': '2026-10-02', 'days': 5}, 'id': 'call_JATCdaxPGIQ9QQPnoR0p4k7W', 'type': 'tool_call'}, {'name': 'to_krw', 'args': {'amount': 30, 'currency': 'USD'}, 'id': 'call_6wUE534kEwlnyDuomoen0z8m', 'type': 'tool_call'}]


In [ ]:
ai.content

In [48]:
messages.append(ai)
for call in ai.tool_calls:
    result = tools_by_name[call["name"]].invoke(call["args"])
    print(f"  ↳ {call['name']}({call['args']}) → {result}")
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

  ↳ add_business_days({'start_date': '2026-10-02', 'days': 5}) → 2026-10-09
  ↳ to_krw({'amount': 30, 'currency': 'USD'}) → {'amount': 30.0, 'currency': 'USD', 'krw': 42000, 'rate': 1400.0}


In [49]:
messages

[HumanMessage(content='2026-10-02부터 영업일 5일 뒤는 며칠이야? 그리고 30달러는 원화로 얼마야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'arguments': '{"start_date":"2026-10-02","days":5}', 'call_id': 'call_JATCdaxPGIQ9QQPnoR0p4k7W', 'name': 'add_business_days', 'type': 'function_call', 'id': 'fc_0906876304d484f9006ae159e86bd487d0982ef0bf35b06fe9', 'status': 'completed'}, {'arguments': '{"amount":30,"currency":"USD"}', 'call_id': 'call_6wUE534kEwlnyDuomoen0z8m', 'name': 'to_krw', 'type': 'function_call', 'id': 'fc_0906876304d484f9006ae159e86bf087d085607292b6644e20', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0906876304d484f9006ae159e6176487d0a55d21567118c396', 'created_at': 1793153510.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0906876304d484f9006ae159e6176487d0a55d21567118c396', tool_calls=[{'name': 'add_business_d

In [50]:
final = llm_with_tools.invoke(messages)
print("\n최종 답변:", final.content)


최종 답변: [{'type': 'text', 'text': '2026년 10월 2일부터 영업일 5일 뒤는 **10월 9일**입니다.  \n**30달러는 42,000원**입니다(환율 1달러 = 1,400원 기준).', 'annotations': [], 'id': 'msg_0906876304d484f9006ae15a98830087d0b2ab6351c4cde656', 'phase': 'final_answer'}]


## Step 04 · 웹 검색 도구

### 4-1. TavilySearch 

- LangChain 공식 통합 (AI용 검색 API)

In [3]:
from langchain_tavily import TavilySearch

tavily = TavilySearch(max_results=3, topic="general")

In [4]:
res = tavily.invoke({"query": "LangChain 1.0 release"})
for r in res.get("results", []):
    print("-", r["title"], "|", r["url"])

- LangChain & LangGraph 1.0 alpha releases | https://www.langchain.com/blog/langchain-langchain-1-0-alpha-releases
- Changelog - Docs by LangChain | https://docs.langchain.com/oss/python/releases/changelog
- What's new in LangChain v1 - Docs by LangChain | https://docs.langchain.com/oss/python/releases/langchain-v1


In [7]:
res['results'][2]

{'url': 'https://docs.langchain.com/oss/python/releases/langchain-v1',
 'title': "What's new in LangChain v1 - Docs by LangChain",
 'content': "## \u200b See also\n\n Versioning – Understanding version numbers\n Release policy – Detailed release policies\n\nConnect these docs to your agent of choice via MCP for real-time answers.\n\nEdit this page on GitHub or file an issue.\n\nWas this page helpful?\n\nYesNo\n\nChangelog\n\nPreviousWhat's new in LangGraph v1\n\nNext\n\nDocs by LangChain home pagelight logodark logo\n\ngithubxlinkedinyoutube\n\nResources\n\nForumChangelogLangChain AcademyContact Sales\n\nCompany\n\nHomeTrust CenterCareersBlog\n\ngithubxlinkedinyoutube [...] Python\n\n Overview\n\n### Reference\n\n Deep Agents\n LangChain SDK\n LangGraph SDK\n Integrations\n Errors\n\n### Releases\n\n Versioning\n Changelog\n Releases\n\n  + LangChain v1\n  + LangGraph v1\n Migration guides\n\n### Policies\n\n Release policy\n Security\n\n## On this page [...] Most of these are re-expor

### 4-2. DDGS 
- API Key 없이 쓰는 검색을 커스텀 도구로 감싸기

In [ ]:
def test_func():
    """독 스트링"""
    ...

test_func.__name__
test_func.__doc__


()

In [ ]:
from ddgs import DDGS

@tool()
def web_search(query: str, max_results: int = 3) -> str:
    """웹에서 최신 정보를 검색합니다. 뉴스 · 최신 동향 · 모델이 모르는 사실을 확인할 때 사용하세요.
    결과는 제목, URL, 요약으로 반환됩니다."""
    try:
        hits = DDGS().text(query, max_results=max_results)
    except Exception as e:
        return f"검색 실패: {e}"
    if not hits:
        return "검색 결과가 없습니다."
    return "\n\n".join(f"[{h['title']}]({h['href']})\n{h['body']}" for h in hits)

# 로직 예시
# def tool(_f, args):
#     name = _f.__name__
#     description = _f.__doc__
#     call_args = _f

#     _f.name = name

#     _f.invoke = lambda args: _f(args)
#     _f.stream = ...

print(web_search.invoke({"query": "OpenAI GPT-6 출시"})[:500])

[OpenAI GPT 6 Astra](https://en.wikipedia.org/wiki/OpenAI_GPT_6_Astra)
GPT-6 is a family of large language models (LLM) developed by American artificial intelligence firm OpenAI. GPT-6 Astra was released to the general public on September 4, 2026, while GPT-6 Sol and GPT-6 Luna were released on September 22, 2026. On October 7, 2026, GPT-6 Instant was released to paid users, featuring an "Intelligent UI" that includes interactive widgets with answers. This was released to free users a day later.


In [ ]:
# tool 데코레이터가 만들어준 애
web_search.name
web_search.description

'웹에서 최신 정보를 검색합니다. 뉴스 · 최신 동향 · 모델이 모르는 사실을 확인할 때 사용하세요.\n결과는 제목, URL, 요약으로 반환됩니다.'

### 4-3. Serper API (Google)

- [Serper](https://serper.dev/)에서 회원가입
     - 초기 2500 크레딧 무료

In [16]:
load_dotenv()

True

In [ ]:
SERPER_API_KEY = os.getenv('SERPER_API_KEY')

In [18]:
from langchain_community.utilities import GoogleSerperAPIWrapper

# type 파라미터를 통해 news, images, places를 검색가능
search = GoogleSerperAPIWrapper(serper_api_key=SERPER_API_KEY)

C:\Users\student\AppData\Local\Temp\ipykernel_3752\2895684901.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import GoogleSerperAPIWrapper


In [19]:
query = "2026년 파이썬 최신 프레임워크 트렌드"
result = search.run(query)

In [ ]:
print(f"검색 결과:\n{result}")

In [28]:
# 구글 이미지 검색

import pprint
search = GoogleSerperAPIWrapper(type="images", serper_api_key=SERPER_API_KEY)
results = search.results("Lion")
pprint.pp(results)

{'searchParameters': {'q': 'Lion',
                      'gl': 'us',
                      'hl': 'en',
                      'type': 'images',
                      'num': 10,
                      'engine': 'google'},
 'images': [{'title': 'Lion - Wikipedia',
             'imageUrl': 'https://upload.wikimedia.org/wikipedia/commons/a/a6/020_The_lion_king_Snyggve_in_the_Serengeti_National_Park_Photo_by_Giles_Laurent.jpg?utm_source=en.wikipedia.org&utm_campaign=index&utm_content=original',
             'imageWidth': 5168,
             'imageHeight': 3448,
             'thumbnailUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRb92CjXX6GH78BZ6Jb_zTzbTQtskWYzHXa3pvkzawzSg&s=10',
             'thumbnailWidth': 547,
             'thumbnailHeight': 365,
             'source': 'Wikipedia',
             'domain': 'en.wikipedia.org',
             'link': 'https://en.wikipedia.org/wiki/Lion',
             'googleUrl': 'https://www.google.com/imgres?imgurl=https%3A%2F%2Fupload.wikimedi

In [24]:
# 구글 뉴스 검색

search = GoogleSerperAPIWrapper(type="news", serper_api_key=SERPER_API_KEY)
results = search.results("Tesla Inc.")
pprint.pp(results)

{'searchParameters': {'q': 'Tesla Inc.',
                      'gl': 'us',
                      'hl': 'en',
                      'type': 'news',
                      'num': 10,
                      'page': 1,
                      'allowSuggested': True,
                      'engine': 'google'},
 'news': [{'title': 'Tesla’s AI ambitions take flight as rivals race to keep '
                    'up',
           'link': 'https://www.ft.com/content/c8db9b39-1922-41f3-b8e9-4488d16f8a2c?syn-25a6b1a6=1',
           'snippet': "SpaceX merger could give fresh thrust to carmaker's "
                      'push into autonomous driving and software as Musk seeks '
                      'to unlock new profits.',
           'date': '1 day ago',
           'source': 'Financial Times',
           'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRxTVlYscEXnwg4p2_aYjHuJa9vPjTxqZLhW_GaFALwqyEEQa7WuiZjELJsn1F7lMgOpXMj77wimEegHUCmEMdWlsSchR-gWmI80_OkJ1mK3u0&s'},
          {'title': '

In [ ]:
# tbs 파라미터를 활용해 검색 결과의 시간을 제한 가능
# qdr:h (past hour) qdr:d (past day) qdr:w (past week) qdr:m (past month) qdr:y (past year)

# 특정 기간을 설정하는것도 가능
# qdr:h12 (past 12 hours)
# qdr:d3, qdr:w2, qdr:m6, qdr:y2
search = GoogleSerperAPIWrapper(type="news", tbs="qdr:h", serper_api_key=SERPER_API_KEY)
results = search.results("Tesla Inc.")
pprint.pp(results)

In [27]:
# 구글 장소 검색

search = GoogleSerperAPIWrapper(type="places", serper_api_key=SERPER_API_KEY)
results = search.results("Italian restaurants in Upper East Side")
pprint.pp(results)

{'searchParameters': {'q': 'Italian restaurants in Upper East Side',
                      'gl': 'us',
                      'hl': 'en',
                      'type': 'places',
                      'num': 10,
                      'page': 1,
                      'engine': 'google'},
 'places': [{'position': 1,
             'title': "Tony's Di Napoli",
             'address': '1081 3rd Ave',
             'latitude': 40.76436,
             'longitude': -73.96424,
             'rating': 4.6,
             'ratingCount': 3300,
             'priceLevel': '$20–70',
             'category': 'Italian',
             'website': 'http://www.tonysnyc.com/',
             'cid': '1820798058748048984'},
            {'position': 2,
             'title': 'Piccola Cucina Uptown',
             'address': '106 E 60th St',
             'latitude': 40.76325,
             'longitude': -73.96898,
             'rating': 4.6,
             'ratingCount': 2500,
             'priceLevel': '$30–80',
             '

### 4-4. 검색도구 Chaining

In [29]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [30]:
template = """You are an AI assistant. Answer the question.
If you don't know the answer, just say you don't know.

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)
parser = StrOutputParser()

chain = prompt | llm | parser

In [31]:
# gpt 6 luna의 지식 단절은 May 18, 2026
result = chain.invoke({"question": "When is the release date for the GPT-6.1 Sol model?"})
print(result)

There’s no publicly confirmed release date for a GPT-6.1 Sol model.


In [32]:
template = """Answer the question based on context.

Question: {question}
Context: {context}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)
parser = StrOutputParser()

chain = (
    {"question": RunnablePassthrough(), "context": RunnablePassthrough()}
    | prompt 
    | llm 
    | parser
)

In [33]:
# 검색 결과 확인
search_result = tavily.run("When is the release date for the GPT-6.1 Sol model?")

print(search_result)

{'query': 'When is the release date for the GPT-6.1 Sol model?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://evolink.ai/blog/gpt-6-1-sol-release-date', 'title': 'GPT-6.1 Sol Release Date: Sept 29, 2026 & Rollout', 'content': "7 min read\n\nGPT-6.1 Sol was released on September 29, 2026, at OpenAI DevDay. The official recap and model reference establish the release and public API identity. For teams already running GPT-6 Sol agents, the immediate question is how the new model changes their existing workflow: ordinary input and output list rates stay the same, cached reads cost less, and the tool-calling contract changes. [...] EvoLink Team\n\n•\n\n18 minEditorial illustration of a model passing a verification gate for the GPT-6 Sol release\n\nmodel-release\n\n### GPT-6 Sol Release Date: Out Sept 22, 2026 — OpenAI's Coding-Agent Model, Rollout by Channel\n\nOpenAI released GPT-6 Sol on September 22, 2026: its model for complex coding and agent 

In [34]:
question = "When is the release date for the GPT-6.1 Sol model?"

result = chain.invoke({"question": question, "context": tavily.run(question)})
print(result)

GPT-6.1 Sol was released on September 29, 2026, at OpenAI DevDay.


## Step 05 · 데이터베이스 쿼리 도구

예제용 SQLite 인메모리 DB 
- 조회(SELECT)만 허용하는 가드 포함

In [51]:
import sqlite3

# 예제용 데이터베이스 셋팅
conn = sqlite3.connect(":memory:", check_same_thread=False)

conn.executescript("""
CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price INTEGER, stock INTEGER);
CREATE TABLE orders (id INTEGER PRIMARY KEY, product_id INTEGER, qty INTEGER, order_date TEXT);
INSERT INTO products VALUES
 (1,'무선 이어폰','전자기기',89000,12),
 (2,'스마트워치','전자기기',199000,0),
 (3,'텀블러','생활용품',29000,40),
 (4,'러닝화','스포츠',119000,7),
 (5,'요가매트','스포츠',35000,25);
INSERT INTO orders VALUES
 (1,1,2,'2026-09-01'),
 (2,3,5,'2026-09-03'),
 (3,4,1,'2026-09-10'),
 (4,1,1,'2026-09-15'),
 (5,5,3,'2026-09-20');
""")

In [52]:
@tool
def list_tables() -> str:
    """데이터베이스의 테이블 목록과 각 테이블의 컬럼을 반환합니다. SQL을 작성하기 전에 먼저 호출하세요."""
    rows = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    out = []
    for (name,) in rows:
        cols = [c[1] for c in conn.execute(f"PRAGMA table_info({name})")]
        out.append(f"{name}({', '.join(cols)})")
    return "\n".join(out)

In [53]:
@tool
def run_sql(query: str) -> str:
    """SQLite SELECT 쿼리를 실행하고 결과를 반환합니다. 조회(SELECT)만 가능하며 최대 20행을 반환합니다."""
    q = query.strip().rstrip(";")
    if not q.lower().startswith("select") or ";" in q:
        return "오류: 단일 SELECT 쿼리만 실행할 수 있습니다."
    try:
        cur = conn.execute(q)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchmany(20)
    except sqlite3.Error as e:
        return f"SQL 오류: {e}. 테이블 · 컬럼 이름을 list_tables로 확인하세요."
    return json.dumps([dict(zip(cols, r)) for r in rows], ensure_ascii=False)

In [54]:
print(list_tables.invoke({}))

products(id, name, category, price, stock)
orders(id, product_id, qty, order_date)


In [55]:
print(run_sql.invoke({"query": "SELECT name, stock FROM products WHERE stock = 0"}))
print(run_sql.invoke({"query": "DELETE FROM products"})) # 가드 확인

[{"name": "스마트워치", "stock": 0}]
오류: 단일 SELECT 쿼리만 실행할 수 있습니다.


## Step 06 · 날씨 조회 API 도구 

- Open-Meteo (API Key 불필요)

In [56]:
import requests

WEATHER_CODE = {0: "맑음", 1: "대체로 맑음", 2: "구름 조금", 3: "흐림", 45: "안개", 51: "이슬비",
                61: "비", 63: "비", 65: "강한 비", 71: "눈", 80: "소나기", 95: "뇌우"}

In [57]:
class WeatherInput(BaseModel):
    city: str = Field(description="도시 이름 (영문 권장, 예: Seoul, Busan, Tokyo)")

In [58]:
@tool(args_schema=WeatherInput)
def get_weather(city: str) -> dict:
    """도시의 현재 기온 · 날씨 · 풍속을 조회합니다."""
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                       params={"name": city, "count": 1, "language": "ko"}, timeout=10).json()
    if not geo.get("results"):
        return {"error": f"'{city}' 위치를 찾을 수 없습니다. 영문 도시 이름으로 다시 시도하세요."}
    loc = geo["results"][0]
    cur = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": loc["latitude"], "longitude": loc["longitude"],
        "current": "temperature_2m,weather_code,wind_speed_10m",
    }, timeout=10).json()["current"]
    return {
        "city": loc["name"], "country": loc.get("country"),
        "temperature_c": cur["temperature_2m"],
        "weather": WEATHER_CODE.get(cur["weather_code"], f"code {cur['weather_code']}"),
        "wind_kmh": cur["wind_speed_10m"],
    }

In [59]:
print(get_weather.invoke({"city": "Seoul"}))

{'city': '서울특별시', 'country': '대한민국', 'temperature_c': 7.6, 'weather': '맑음', 'wind_kmh': 5.2}


## Step 07 · 에이전트 - create_agent

### 7-1. 도구를 모두 연결한 에이전트

In [60]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[web_search, list_tables, run_sql, get_weather, to_krw],
    system_prompt=(
        "당신은 쇼핑몰 운영을 돕는 어시스턴트입니다. "
        "재고 · 주문은 데이터베이스 도구로, 날씨는 날씨 도구로, 최신 정보는 웹 검색으로 확인하세요. "
        "도구 결과에 근거해 한국어로 간결하게 답하세요."
    ),
)

In [61]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "재고가 10개 미만인 상품과 가격을 알려줘"}
    ]})

print(result["messages"][-1].content)

[{'type': 'text', 'text': '재고가 10개 미만인 상품은 다음과 같습니다.\n\n- 스마트워치 — 199,000원 (재고 0개)\n- 러닝화 — 119,000원 (재고 7개)', 'annotations': [], 'id': 'msg_08e17ec6ad0281aa006ae15b7ca78487d0b38ef5ca338c2617', 'phase': 'final_answer'}]


In [62]:
result

{'messages': [HumanMessage(content='재고가 10개 미만인 상품과 가격을 알려줘', additional_kwargs={}, response_metadata={}, id='63b5e6e2-60a9-4b60-9b5f-5fe9906f2432'),
  AIMessage(content=[{'arguments': '{}', 'call_id': 'call_Pq3HP558ZWm9YGh4GLFSANNP', 'name': 'list_tables', 'type': 'function_call', 'id': 'fc_08e17ec6ad0281aa006ae15b7924ec87d0a95a5cd6e4644b22', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_08e17ec6ad0281aa006ae15b77f1d487d0a54b160e8877c183', 'created_at': 1793153911.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_08e17ec6ad0281aa006ae15b77f1d487d0a54b160e8877c183', tool_calls=[{'name': 'list_tables', 'args': {}, 'id': 'call_Pq3HP558ZWm9YGh4GLFSANNP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 356, 'output_tokens': 14, 'total_tokens': 370, 'input_token_details': {'cache_creation': 0, 'cache_r

In [63]:
# 에이전트가 거친 단계 확인
for m in result["messages"]:
    kind = type(m).__name__
    if getattr(m, "tool_calls", None):
        print(f"{kind}: tool_calls → {[(c['name'], c['args']) for c in m.tool_calls]}")
    else:
        print(f"{kind}: {str(m.content)[:100]}")

HumanMessage: 재고가 10개 미만인 상품과 가격을 알려줘
AIMessage: tool_calls → [('list_tables', {})]
ToolMessage: products(id, name, category, price, stock)
orders(id, product_id, qty, order_date)
AIMessage: tool_calls → [('run_sql', {'query': 'SELECT name, price, stock FROM products WHERE stock < 10 ORDER BY stock ASC;'})]
ToolMessage: [{"name": "스마트워치", "price": 199000, "stock": 0}, {"name": "러닝화", "price": 119000, "stock": 7}]
AIMessage: [{'type': 'text', 'text': '재고가 10개 미만인 상품은 다음과 같습니다.\n\n- 스마트워치 — 199,000원 (재고 0개)\n- 러닝화 — 119,000원


### 7-2. 스트리밍 

- 단계별 진행 상황 보기

In [64]:
question = {
    "messages": [
        {"role": "user", "content": "서울 날씨를 보고, 스포츠 카테고리 상품 중 오늘 추천할 만한 것을 골라줘"}
    ]}

for chunk in agent.stream(question, stream_mode="updates"):
    for node, update in chunk.items():
        last = update["messages"][-1]
        if getattr(last, "tool_calls", None):
            print(f"[{node}] 도구 호출:", [c["name"] for c in last.tool_calls])
        else:
            print(f"[{node}]", str(last.content)[:150])

[model] 도구 호출: ['get_weather']
[tools] {"city": "서울특별시", "country": "대한민국", "temperature_c": 7.6, "weather": "맑음", "wind_kmh": 5.2}
[model] 도구 호출: ['list_tables']
[tools] products(id, name, category, price, stock)
orders(id, product_id, qty, order_date)
[model] 도구 호출: ['run_sql']
[tools] [{"id": 5, "name": "요가매트", "price": 35000, "stock": 25}, {"id": 4, "name": "러닝화", "price": 119000, "stock": 7}]
[model] [{'id': 'rs_058ddde8a041696b006ae15c72933887d0809edd48513336d4', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq4Vx0


### 7-3. 메모리 

- checkpointer + thread_id

In [65]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model=llm, tools=[get_weather],
    system_prompt="당신은 여행 도우미입니다.",
    checkpointer=InMemorySaver(),
)

In [66]:
cfg = {"configurable": {"thread_id": "trip-001"}}
memory_agent.invoke({"messages": [{"role": "user", "content": "다음 주에 부산 여행 가요."}]}, cfg)

{'messages': [HumanMessage(content='다음 주에 부산 여행 가요.', additional_kwargs={}, response_metadata={}, id='4c599ea3-845c-4008-ba19-bf1586399844'),
  AIMessage(content=[{'id': 'rs_05fcc6966fc50f2a006ae15cefa5fc87d0941b02e96c80e5cd', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq4VzxNy075VciPfoKlZTY58D_XrPhXVpeD_OC9tkA-OAhdAaJV_XzCI2H_z2R1VdAvxKZ6jXCQ4Z8KTMfkSCaWTBRlsRn27Js99KSICzeIQQJHOk20eJqA_-upZLzwNNhA-dj21livAADjNlewv-Xf7NAbDITaBFdtc03KKHey5NCMn2FjQ08patfBjwTQFZI_sO5oHZELTirautn5cWbjcGpUuSPBZKDdWsFvchlMXyyNpDXGchdsswpfSO-6GYcGDdZvOaoZUAJMG3UWIicroOIUt4c83h3VkVdeEwNjtGlM4CSF-nJjEKfnEh4vJk-HyfBit4wu_iGNryZpyaNL7VcUwbh33eK_g7a0-kFubmRpZPzGS33ME1s5cbAemo1GS4s_o0IvKjxN71YNfbtyxCiIzhOGwNKyPiK0pkOtAW6hHmW4lId_26ZFwByOcESP42d27alfVZdGsx54SVOtcscWLr9CpDtgLsOy2IQHSx92Y4LuNA3ZeBb9AIPkcpYswO0KV-sll2Y3lYIqBqVGCL6F9XrMqc6_XeAmMAxa75Sjwb-EvINJW6KDL-EUzLOsUmpoMOlcekEWE8NgT-kzrLTGkCNWK4xJXOwLas4XVCUBaTa9ND5930uRHOlQt8fw8ChayUc-BfDbQ5JO1yYB5mV6vCVdYJJ6IRzGrL0rdcex27aqBrD

In [67]:
r = memory_agent.invoke({"messages": [{"role": "user", "content": "거기 지금 날씨 어때?"}]}, cfg)
print(r["messages"][-1].content)

[{'type': 'text', 'text': '부산은 지금 **맑고 12.4°C**, 바람은 **시속 9.1km** 정도예요. 꽤 선선하니 가벼운 겉옷을 챙기면 좋겠어요.', 'annotations': [], 'id': 'msg_05fcc6966fc50f2a006ae15d2df13887d092fca21822f58268', 'phase': 'final_answer'}]


In [ ]:
r['messages']

In [69]:
other = memory_agent.invoke({"messages": [{"role": "user", "content": "거기 지금 날씨 어때?"}]},
                            {"configurable": {"thread_id": "trip-002"}})
print("\n[다른 thread]", other["messages"][-1].content)


[다른 thread] [{'id': 'rs_04ee36536654e134006ae15d7388c487d0a073a61da50049ba', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq4V10UemNVLCSCn5qi4-Z-rlSNYlPlrLqEhruw0flksNwmU24UjKL4EJgiX2KSHCqQGo4nPfRUoTh9z7_uAY-TilDBani1T7uzTFNxRE_XafNF7f_zTqcjb-bOkqmfNj8Av6pTmS5ZKe54qH-RhOOGTlw6x-sndYGy3KAXx6kNjwTBBX_v5F01Qnbjn29s-BNYbQYmUconQQd3DmAoGdWxbbwaKfE51_IJQKE0U89zxOUVZ3qsm4OAIoS6GtGaPvQrhtLpv7TqUN_mu0J6OhPCc5Y9EmGZDf_yXN-8rcsLHrYs6z9hmovbaU6DYJK_CRQ50UIoMMFGRHKXpo6oZOf0VLbdRKb1GoUAweAUcLj8TqQ6AsRLVOu8BBB9QyCApZjPjOmwWJ5sfIrjh3t5RoN2CnKbz--KQVxOvFLR6X9SKWL6gKo8jdTkfGRlQHVt40OV_V3IAwjaDIS1z9JeIB-GP0iLt4EZ4jcoUJwwrWis6VD9aCs7GWEpvPR0WBwjQU332rCLpxo5wY21w6Je00teDSpweHBt6zZlkx_fW1ioGejNdwjlxWMenO90qNwjBs6a6KbE7ThWnsVFQYuN10hKvx-ehOS0LV03ifGtDUae7zW4eMfUKhZdxsXD4dBpOdwOGGL-Op4FQMPfnGFphkAL1XF3KYDUOvKVbjt4Mj3C2p6gMYOwAtx0p4ekk8ikEJaB_v0uCe4COszNqPCTRBs0iJ1erOwW9jjNVRLBNAw3HIqnJD8r8UeRmwHPjHR90kJWZefyg_lzUx4iTOVWYpRwXpgiMACe4gv14DjLyoB0dJxXqD3_TJzFTTNldwxrz9RScR_fAT-Z

## Step 08 · 고급

### 8-1. ToolRuntime 
- 모델에게 보이지 않는 실행 정보(context) 사용


In [70]:
from dataclasses import dataclass
from langchain.tools import ToolRuntime

@dataclass
class UserContext:
    user_id: str
    grade: Literal["basic", "vip"]

In [71]:
@tool
def my_discount(price: int, runtime: ToolRuntime[UserContext]) -> str:
    """현재 사용자의 회원 등급 할인을 적용한 가격을 계산합니다."""
    rate = 0.1 if runtime.context.grade == "vip" else 0.0
    return f"{runtime.context.user_id}({runtime.context.grade}) 할인가: {int(price * (1 - rate)):,}원"

print("모델에 보이는 인자:", my_discount.args)   # runtime은 숨겨짐

모델에 보이는 인자: {'price': {'title': 'Price', 'type': 'integer'}}


In [72]:
ctx_agent = create_agent(
    model=llm, 
    tools=[my_discount, run_sql], 
    context_schema=UserContext,
    system_prompt="가격은 DB에서 조회하고, 할인은 my_discount로 계산하세요."
)

In [73]:
r = ctx_agent.invoke({
    "messages": [{"role": "user", "content": "러닝화 내 할인가 알려줘"}]},
    context=UserContext(user_id="u123", grade="vip")
)
print(r["messages"][-1].content)

[{'type': 'text', 'text': '러닝화의 정가는 119,000원이며, 회원 할인가격은 **107,100원**입니다.', 'annotations': [], 'id': 'msg_02cec928a67aa1d5006ae15e0d5e3087d09795e9df79d98c81', 'phase': 'final_answer'}]


In [74]:
r['messages']

[HumanMessage(content='러닝화 내 할인가 알려줘', additional_kwargs={}, response_metadata={}, id='45df82d0-9ea7-459e-bc3b-d075131b163e'),
 AIMessage(content=[{'id': 'rs_02cec928a67aa1d5006ae15e09167087d0b26bd59acc998a13', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq4V4KL5TBgm6etcHyj8OKPWhneWKCIhlKs8a60D-hyMTbnLD7p9ZRpet04yJ-kXGfBlA9h6qJ1vbVZfgNCZ17n8EW3g0S9P1JtTE00OgH4YRAE01g52nxQMKHuP_C7vZfesQ-bVuiwbR1sPn2jZe_5SLf6gLkPVuC7m70zH3DEBD7_jv-evKaAldnDT60x8zX6JrfAiCuHnVdT0mVJ_1XipKBRdngvygHuhI2yLfMY3JJmvfg20kcxkuhrUpRHn8LCWwaNXgMWG3Zl1u4B-LBfEUkjs2XyqKuy7FYOVZZJamknUd1AtBVElcHDsBodMp-Rt1yeR-Z4aj2ETwdH0B7q6pGVOuiFu9tMl4QzgoCu38yB93fz00nUjFwuB_wiHbXiN_jvgEkoIyaipusWV3JEwnvHBqBuS3b6H8HqzHf_HdOgJ_yYka1-8n9Wzwa16lQ_rNuM0IteBSTBA1JcP2BHnUSvjr88yV7gnXrvC4eoyxPIGa5p-3SBN2G9FmnjzdX4SNjWmMr8NrpGzEq6bKZ8auyZLk6cHI5MlWCIlQpp26IPn-v-2cHTMbeT191x6aUnqpwjrdPP_IenavXKXviKO81WFt9UOpaQQS1LE8SPAfd5EWe2uj2prgZagMq3nAWS3rWnS6uwE8afTOVZQ4yHfha7bKk5Qhxh7ynvFiBkVutlQxoKmvWBdDWjWglqM1dkyzT

### 8-2. 구조화 출력 — response_format

In [75]:
class WeatherBrief(BaseModel):
    city: str
    temperature_c: float
    outfit: str = Field(description="기온에 맞는 옷차림 추천 한 문장")

In [76]:
brief_agent = create_agent(model=llm, tools=[get_weather], response_format=WeatherBrief)

In [77]:
r = brief_agent.invoke({"messages": [{"role": "user", "content": "도쿄 날씨 기준으로 옷차림 추천해줘"}]})
print(r["structured_response"])

city='도쿄' temperature_c=17.5 outfit='맑고 17.5°C로 선선하니 긴팔 상의에 가벼운 재킷이나 가디건을 걸치면 좋습니다.'


### 8-3. 미들웨어

- 도구 재시도 · 호출 횟수 제한

In [78]:
from langchain.agents.middleware import ToolCallLimitMiddleware, ToolRetryMiddleware

safe_agent = create_agent(
    model=llm,
    tools=[web_search, get_weather],
    middleware=[
        ToolRetryMiddleware(max_retries=2),                  # 도구 실패 시 재시도
        ToolCallLimitMiddleware(tool_name="web_search", run_limit=3),  # 검색은 1회 실행당 최대 3번
    ],
)

In [79]:
r = safe_agent.invoke({"messages": [{"role": "user", "content": "LangChain과 LangGraph의 차이를 최신 자료로 정리해줘"}]})
print(r["messages"][-1].content[:500])

[{'id': 'rs_0debbd64765eccea006ae15ea287ec87d0a1f488f38082cebe', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABq4V6s93vk6SN7uEDOUlxPCLzWEeWd2YeKaJDq7SFlGwNQIVSJDUhx0YAlEXoF7Jl4iO-6Qs6dNicwmOQBoC_COiehKVYwjK6wxOjuMWrg54ZEVwKhtrZ29O7O-rj2Fo22ywxGJtLzg_JNTVdaKUvKjbGzKtcuVHzuvR58OPpalcZp4bNGerQa9U9_4sAT1BOfc72cQsQMX_fotJSEX0Fa3ff9BHTSznN3_JQ3c5bZ8qyeNZpgjtfAOC37d7C0wWH4SUoKqBmofxXlJfl1isNuhGgly2p5TP1M219UtFBwypZF_ba76Dl9D61JVt1rVNRcQ7eL4tDRzRVH7ViPRHplAJqUn0RdspQqnHRDVoWAsJOQRGVm1hjXiXgjamhZOnfFDPy9JCsfPebqYcsY3W7qmB02eK6Apg3SPgSyJknkSaLN7Ezsz56D3n7DcUATGqxH-7qjGt3R_7tvJzIOeaznViF3K5Qddy2s1kk_AYITgj970sdSLNT6DHiO4KoqSNOHlEslvU2OladHZakmdbB_0p7jyYj5WeZzZLg-BKyw3TP2duizWFBXQn3fXk2jATEIKnJesWFwgmG_uGX84UtRxW3cf5SHEVn2TwdBEHMOMb3U_77RcYQ6CiZYzGJbZZJVMNnpkFhlfp-nJnTaIMq_akzwzg1kEuhWYhhvjR8vASircn59VSn-2LA0Bj81Ox7HcHac0wizgRuDrvuGBpyLrSV_oDUTt8Pqo45TQ6LPlaHdATlQK4neoAlYPswCKsoJTo2m34Mi1RxA6iOFL5SWC9q1-5hbr9ICFdDm5lQ2AkihW2odBeWJo8dTeq1ZwxB7vQ4_ua5tXemP0_o0TKmYHr

In [87]:
print(r['messages'][-1].content[-1]['text'])

## 한 줄 요약

**LangChain은 LLM 앱과 에이전트를 빠르게 만드는 상위 수준 프레임워크**, **LangGraph는 에이전트·워크플로의 실행 흐름과 상태를 세밀하게 제어하는 오케스트레이션 런타임**입니다. 둘은 대체재라기보다 함께 쓰는 관계입니다.

| 구분 | LangChain | LangGraph |
|---|---|---|
| 주된 역할 | 모델·도구를 연결하고 에이전트를 빠르게 구성 | 상태가 있는 에이전트와 복잡한 작업 흐름을 설계·실행 |
| 추상화 수준 | 높음. 미리 준비된 에이전트 구성과 통합 제공 | 낮음. 상태, 노드, 엣지 등으로 흐름을 직접 정의 |
| 적합한 경우 | 일반적인 도구 사용 에이전트나 LLM 앱을 빠르게 만들 때 | 분기·반복이 복잡하거나, 장시간 실행·재개·사람의 승인이 필요한 경우 |
| 주요 기능 | 모델·도구 통합, 에이전트 구성, 미들웨어 | 상태 관리, 체크포인트·재개, 스트리밍, 사람 개입, 사용자 정의 흐름 |

### 핵심 차이

- **LangChain**은 표준적인 에이전트 구조를 쉽게 구성하도록 돕습니다. LangChain v1에서는 `create_agent`가 대표적인 에이전트 생성 방식입니다.
- **LangGraph**는 에이전트가 *어떤 순서로 무엇을 실행할지*를 더 직접 다룹니다. 상태를 가진 노드와 전이로 흐름을 만들고, 실행을 저장·재개하거나 사람의 검토를 끼워 넣을 수 있습니다.
- 둘은 경쟁 관계가 아닙니다. **LangChain의 에이전트는 LangGraph 위에서 실행**되므로, 간단한 시작은 LangChain으로 하고 세밀한 제어가 필요해지면 LangGraph로 내려가는 방식이 가능합니다.

### 어떤 것을 선택할까?

- **일반적인 챗봇·도구 사용 에이전트를 빨리 구현**하고 싶다면 → **LangChain**
- **업무 단계가 고정된 워크플로**, 복잡한 분기·반복, 장기 실행, 중단 후 재개, 승인 절차가 필요하다면 → **LangGraph**
- Lang